In [24]:
import pandas as pd

df=pd.read_csv("../data/catalog/dados_merge.csv",encoding="latin1", sep=";", low_memory=False)

df.columns = df.columns.str.replace(r'\s+', ' ', regex=True).str.strip()

coluna_aprov = next((c for c in df.columns if "Aprov" in c or "aprov" in c), None)
coluna_reprov = next((c for c in df.columns if "Reprov" in c or "reprov" in c), None)
coluna_aband = next((c for c in df.columns if "Aband" in c or "aband" in c), None)


colunas_encontradas = [c for c in [coluna_aprov, coluna_reprov, coluna_aband] if c is not None]



for col in colunas_encontradas:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r'\s+', '', regex=True)
        .str.replace("--", "nan")
        .str.replace(",", ".")
        .astype(float)
    )

df.rename(columns={"AprovaÃ§Ã£o": "Aprovação", "ReprovaÃ§Ã£o": "Reprovação"}, inplace=True)

df["Internet"]=(
    df["IN_INTERNET_ALUNOS"]+
    df["IN_INTERNET_APRENDIZAGEM"]
)

df["Indice_Recursos"]=(
    df["Internet"]+
    df["IN_LABORATORIO_INFORMATICA"]+
    df["IN_BIBLIOTECA"]
)

df["Indice_Tecnologia"]=(
    df["Internet"]+
    df["IN_LABORATORIO_INFORMATICA"]
)

df["Indice_Infraestrutura"]=(
    df["IN_BANHEIRO"]+
    df["IN_AGUA_POTAVEL"]+
    df["Energia"]+
    df["Esgoto"]
)

df["Indice_Informação"]=(
    df["Internet"]+
    df["IN_BIBLIOTECA"]
)

df["Indice_Acessibilidade"]=(
    df["IN_BANHEIRO_PNE"]+
    df["IN_ACESSIBILIDADE_CORRIMAO"]+
    df["IN_ACESSIBILIDADE_ELEVADOR"]+
    df["IN_ACESSIBILIDADE_VAO_LIVRE"]+
    df["IN_ACESSIBILIDADE_RAMPAS"]+
    df["IN_ACESSIBILIDADE_SINAL_SONORO"]+
    df["IN_ACESSIBILIDADE_SINAL_TATIL"]+
    df["IN_ACESSIBILIDADE_SINAL_VISUAL"]
)

In [27]:
#Tipo de localização -> acesso a recursos
recursos_localizacao=(
    df.groupby("TP_LOCALIZACAO")["Indice_Recursos"]
    .mean()
    .round(2)
    .reset_index()
)

#rendimento -> tecnologia
rendimento_tecnologia=(
    df.groupby("NO_REGIAO")
    .agg({
        "Indice_Tecnologia": "mean",
        "Aprovação": "mean",
        "Reprovação": "mean"
    })
    .round(2)
    .reset_index()
)

#rendimento e abandono -> infraestrutura
infra_rendimento=(df.groupby("NO_REGIAO")
    .agg({
        "Indice_Infraestrutura": "mean",
        "Aprovação": "mean",
        "Abandono": "mean"
    })
    .round(2)
    .reset_index()
)

#rendimento -> acesso à informação
informacao_rendimento=(
    df.groupby("NO_REGIAO")
       .agg({
        "Indice_Informação": "mean",
        "Aprovação": "mean",
        "Reprovação": "mean"
    })
    .round(2)
    .reset_index()
)

#rendimento e abandono -> acesso à recursos
recursos_rend_abandono=(
    df.groupby("NO_REGIAO")
       .agg({
        "Indice_Recursos": "mean",
        "Aprovação": "mean",
        "Abandono": "mean"
    })
    .round(2)
    .reset_index()
)

#abandono -> acessibilidade
abandono_acessibilidade=(
    df.groupby("NO_REGIAO")
      .agg({
          "Indice_Acessibilidade":"mean",
          "Abandono":"mean"
      })
      .round(2)
      .reset_index()
)

In [29]:
#correlações

#tecnologia
corr_tecnologia_rendimento=(
    df["Indice_Tecnologia"]
    .corr(df["Aprovação"])
)

#infraestrutura
corr_infra_rend=(
    df["Indice_Infraestrutura"]
    .corr(df["Aprovação"])
)

corr_infra_ab=(
    df["Indice_Infraestrutura"]
    .corr(df["Abandono"])
)

#acesso à informação
corr_info_rend=(
    df["Indice_Informação"]
    .corr(df["Aprovação"])
)


#acesso a recursos
corr_rec_ren=(
    df["Indice_Recursos"]
    .corr(df["Aprovação"])
)

corr_rec_ab=(
    df["Indice_Recursos"]
    .corr(df["Abandono"])
)

#acessibilidade
corr_acess_ab=(
    df["Indice_Acessibilidade"]
    .corr(df["Abandono"])
)

In [33]:
dados_esquematizados={
    "Relação analisada": [
        "Tecnologia x Rendimento",
        "Infraestrutura x Rendimento",
        "Infraestrutura X Abandono",
        "Acesso à Informação X Rendimento",
        "Acesso a Recursos X Rendimento",
        "Acesso a Recursos X Abandono",
        "Acessibilidade X Abandono"
    ],
    "Correlação de Pearson": [
        corr_tecnologia_rendimento,
        corr_infra_rend,
        corr_infra_ab,
        corr_info_rend,
        
        corr_rec_ren,
        corr_rec_ab,
        corr_acess_ab
    ]
}

correlacoes=pd.DataFrame(dados_esquematizados)

correlacoes["Correlação de Pearson"]=correlacoes["Correlação de Pearson"].round(3)

correlacoes.head()

,Relação analisada,Correlação de Pearson
0,Tecnologia x Rendimento,0.078
1,Infraestrutura x Rendimento,0.040
2,Infraestrutura X Abandono,-0.057
3,Acesso à Informação X Rendimento,0.051
4,Acesso a Recursos X Rendimento,0.040


In [34]:
df.to_csv("../data/processed/dados_rend_proc.csv", sep=";", decimal=",", encoding="utf-8-sig", index=False)

recursos_localizacao.to_csv("../data/processed/recursos_localizacao.csv", sep=";", decimal=",", encoding="utf-8-sig", index=False)

rendimento_tecnologia.to_csv("../data/processed/rendimento_tecnologia.csv", sep=";", decimal=",", encoding="utf-8-sig", index=False)

infra_rendimento.to_csv("../data/processed/infra_rendimento.csv", sep=";", decimal=",", encoding="utf-8-sig", index=False)

informacao_rendimento.to_csv("../data/processed/informacao_rendimento.csv", sep=";", decimal=",", encoding="utf-8-sig", index=False)

recursos_rend_abandono.to_csv("../data/processed/recursos_rend_abandono.csv", sep=";", decimal=",", encoding="utf-8-sig", index=False)

abandono_acessibilidade.to_csv("../data/processed/abandono_acessibilidade.csv", sep=";", decimal=",", encoding="utf-8-sig", index=False)

correlacoes.to_csv("../data/processed/correlacoes.csv", sep=";", decimal=",", encoding="utf-8-sig", index=False)

print("Arquivos gerados com sucesso!")

Arquivos gerados com sucesso!
